In [1]:
# Library - basics
import warnings
warnings.filterwarnings("ignore", category = UserWarning)
import os
import json
import csv
import torch
import numpy as np
import joblib
import argparse
from datetime import datetime
import pandas as pd

# Library - manual
import building_cognitive_diagram as bcd
import sample_generation as sg
import model_training as mt
import analysis_functions as af

# Library - sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.svm import SVC

## Basics

In [2]:
### Variables ###
task_list = ["WCST", "MaxNum", "Nback", "FiniteSet", "FiniteList", "FiniteStack"]
if_minimal = False

seq_len = 10    # the length of each stimuli sequence
num_train_trials = 50000
num_test_trials = 10000
random_seed = 29
target_cond = {"size": 8, "l2": 1e-4, "kernel": "rbf"}	# None or {"size": 16, "l2": 1e-3}

batch_size = 4
learning_rate = 0.001
val_ratio = 0.2    # validation ratio

tol = 1e-3
eps = 1e-12
num_permutations = 10

In [3]:
### Functions ###
def local_save_spacifications(task_type):

	metadata = {
		"task_type": task_type, "seq_len": seq_len, "num_train_trials": num_train_trials, "num_test_trials": num_test_trials,
		"random_seed": random_seed, "target_cond": target_cond, "batch_size": batch_size,
		"learning_rate": learning_rate, "val_ratio": val_ratio, "tol": tol, "if_minimal": if_minimal
		}

	with open(os.path.join(results_path, f"{task_type}_metadata.json"), "w") as f:
		json.dump(metadata, f, indent = 2)


def local_build_cognitive_diagram(task_type):

	task = bcd.build_cognitive_diagram(task_type, if_minimal)

	with open(os.path.join(results_path, f"{task_type}_task.json"), "w") as f:
		json.dump(task.to_dict(), f, indent = 2)

	return task


def local_generate_samples(task_type):

	train_samples, test_samples, train_ans, test_ans, train_state, test_state, stimuli_encoder, ans_encoder, train_loader, val_loader, test_loader = sg.generate_loaders(task, task_type, seq_len, num_train_trials, num_test_trials, random_seed, val_ratio, batch_size)

	print(f"- train_samples.shape: {np.array(train_samples).shape} | train_ans.shape: {np.array(train_ans).shape}")
	print(f"- test_samples.shape: {np.array(test_samples).shape} | test_ans.shape: {np.array(test_ans).shape} | test_state: {np.array(test_state).shape}")
	np.savez_compressed(os.path.join(results_path, f"{task_type}_test_data.npz"), 
                        train_samples = np.array(train_samples),
						train_ans = np.array(train_ans),
						test_samples = np.array(test_samples),
						test_ans = np.array(test_ans),
						test_state = np.array(test_state)
						)

	joblib.dump({
				"stimuli_encoder": stimuli_encoder,
				"ans_encoder": ans_encoder
				}, os.path.join(results_path, f"{task_type}_encoders.joblib"))

	return (train_samples, test_samples), (train_ans, test_ans), (train_state, test_state), (stimuli_encoder, ans_encoder), (train_loader, val_loader, test_loader)


def local_model_training(train_loader, val_loader, test_loader, task):


	model, log = mt.train_model(train_loader, val_loader, test_loader, task, learning_rate, 
						layer_size = target_cond["size"], regularization = target_cond["l2"], random_seed = random_seed, get_vectors = True)
	
	joblib.dump(log, os.path.join(results_path, f"{task_type}_log.npz"))

	return model, log

def local_extract_sim_level(model, test_loader, test_samples, test_ans, test_state):
	
	test_vectors = mt.extract_features(model, test_loader)
	stim_level_sample, stim_level_vec, stim_level_ans, stim_level_state, stim_level_trial = mt.stim_level_features(test_samples, test_vectors, test_ans, test_state)

	return stim_level_sample, stim_level_vec, stim_level_ans, stim_level_state, stim_level_trial


def local_decodability(stim_level_vec, stim_level_state):

	cv = StratifiedKFold(n_splits = 10, shuffle = True, random_state = random_seed)
	clf = make_pipeline(StandardScaler(), SVC(kernel = target_cond["kernel"], tol = tol))
	acc = cross_val_score(clf, stim_level_vec, stim_level_state, cv = cv, scoring = "accuracy")

	return acc


def local_cond_decodability(stim_test_vec, cond_var_list, decode_var_list):

    cond_acc = {var: dict() for var in cond_var_list}
    stim_test_vec = np.asarray(stim_test_vec)
    
    for var1 in cond_var_list:
        print(f"- conditioned on {var1}")
        
        for var2 in decode_var_list:
    
            if var1 == var2:
                continue
    
            cond_acc[var1][var2] = []
            
            cond_var = np.asarray(other_vars[var1])
            decode_var = np.asarray(other_vars[var2])

            for cond in np.unique(cond_var):
    
                mask = cond_var == cond
    
                stim_test_vec_masked = stim_test_vec[mask]
                
                stim_test_vec_var = np.var(stim_test_vec_masked, axis = 0)
                if np.sum(stim_test_vec_var) < eps:
                    continue
                    
                decode_var_masked = decode_var[mask]
                
                decode_classes, class_counts = np.unique(decode_var_masked, return_counts = True)

                if len(decode_classes) < 2:
                    print(f"--- {var2} < 2 classes ({len(decode_classes)}) conditioned on {var1} {cond}")
                    continue
                
                if np.all(class_counts < 10):
                    print(f"--- {var2} < 10 counts in all classed conditioned on {var1} {cond}")
                    continue
                
                acc_temp = local_decodability(stim_test_vec_masked, decode_var_masked)
                cond_acc[var1][var2].extend(acc_temp)
    
            print(f"-- decoded {var2}")

    return cond_acc


def local_separability(stim_level_vec, stim_level_state):

	separability_index = af.calculate_separability(stim_level_vec, stim_level_state, eps = eps)

	return separability_index


def local_get_othervars(test_samples, test_ans, state_list):

	other_vars = af.get_other_vars(test_samples, test_ans, state_list)

	np.savez_compressed(os.path.join(results_path, f"{task_type}_othervars.npz"), **other_vars)

	return other_vars


def local_clustering(vectors, var_list, num_states):

	clustering_idx = af.get_clustering(vectors, var_list, num_states, random_seed)

	return clustering_idx


def local_alignment(test_samples, log, test_state):

	k = 10

	epoch_acc = af.epoch_decodability(test_samples, log, test_state, target_cond["kernel"], k, random_seed)
	#alignment = af.compute_alignment(log, epoch_acc)

    #return alignment
	return epoch_acc


def local_axis_removal(model, test_loader, vectors, stim_test_state):

	acc_abl = af.evaluate_removed_performance(model, test_loader, vectors, stim_test_state, num_permutations, random_seed)

	return acc_abl

def local_state_confusion(task, model, test_samples, stim_test_vec, test_loader, stim_test_state):

    state_vec_dict, state_num_dict = af.state_centroids(stim_test_vec, stim_test_state)
    acc_conf = af.evaluate_conf_performance(task, model, stimuli_encoder, ans_encoder, state_vec_dict, test_loader, test_state, test_ans)

    return acc_conf, state_vec_dict

In [4]:
def to_jsonable(obj):
    
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    elif isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_jsonable(v) for v in obj]
    elif isinstance(obj, tuple):
        return [to_jsonable(v) for v in obj]
    else:
        return obj

In [5]:
date = datetime.now().strftime("%Y%m%d")

for task_type in task_list:
    print("Note: If you want to change the task variables, refer to building_cognitive_diagram.py")
    print(f"task_type: {task_type}")

    # directories
    results_path = os.path.join('results', date, task_type,
    							f"random{random_seed}_size{target_cond['size']}_regul{target_cond['l2']}_kernel{target_cond['kernel']}")
    os.makedirs(results_path, exist_ok = True)

    # basics
    print("= Sample generation =")
    local_save_spacifications(task_type)
    task = local_build_cognitive_diagram(task_type)
    samples, ans, states, encoders, loaders = local_generate_samples(task_type)
    
    train_samples, test_samples = samples
    train_ans, test_ans = ans
    train_state, test_state = states
    stimuli_encoder, ans_encoder = encoders
    train_loader, val_loader, test_loader = loaders

    print("= Model training =")
    model, log = local_model_training(train_loader, val_loader, test_loader, task)
    stim_test_sample, stim_test_vec, stim_test_ans, stim_test_state, stim_test_trial = local_extract_sim_level(model, test_loader, test_samples, test_ans, test_state)

    # decodability
    print("= Decodability & Separability =")
    other_vars = local_get_othervars(test_samples, test_ans, stim_test_state)
    
    acc = dict()
    for var in other_vars.keys():
        
        var_list = other_vars[var]
        acc[var] = local_decodability(stim_test_vec, var_list)
        
        print(f"- {var} decoding done")
    
    with open(os.path.join(results_path, f"{task_type}_decodability.json"), "w") as f:
        json.dump(to_jsonable(acc), f, indent = 2)

    # separability
    sep = dict()
    for var in other_vars.keys():
        
        var_list = other_vars[var]
        sep[var] = local_separability(stim_test_vec, var_list)
        
        print(f"- {var} separability done")
    
    with open(os.path.join(results_path, f"{task_type}_separability.json"), "w") as f:
        json.dump(to_jsonable(sep), f, indent = 2)

    # cond decodability
    print("= Cond Decodability & Separability =")
    cond_var_list = ["last_stim", "sec_last_stim", "last_two_stim", "curr_ans", "last_ans", "trial_idx"]
    decode_var_list = ["cog_state"]
    
    cond_acc = local_cond_decodability(stim_test_vec, cond_var_list, decode_var_list)
    with open(os.path.join(results_path, f"{task_type}_cond_decodability.json"), "w") as f:
        json.dump(to_jsonable(cond_acc), f, indent = 2)

    # cond separabilty
    cond_var_list = ["cog_state"]
    decode_var_list = ["last_stim", "sec_last_stim", "last_two_stim", "curr_ans", "last_ans", "trial_idx"]
    
    rev_cond_acc = local_cond_decodability(stim_test_vec, cond_var_list, decode_var_list)
    with open(os.path.join(results_path, f"{task_type}_rev_cond_decodability.json"), "w") as f:
        json.dump(to_jsonable(rev_cond_acc), f, indent = 2)

    # clustering index
    print("= Clustering index =")
    clu_idx = dict()
    for var in other_vars.keys():
        
        var_list = other_vars[var]
        clu_idx[var] = local_clustering(stim_test_vec, var_list, len(np.unique(stim_test_state)))
        
        print(f"- {var} clustering done")
    
    with open(os.path.join(results_path, f"{task_type}_clustering.json"), "w") as f:
        json.dump(to_jsonable(clu_idx), f, indent = 2)

    # alignment
    print("= Alignment =")
    corr_loss = dict()
    for var in other_vars.keys():
        
        var_list = other_vars[var]
        epoch_acc = local_alignment(test_samples, log, test_state)
        #corr_loss[var] = local_alignment(test_samples, log, test_state)
        
        print(f"- {var} alignment done")
    
    #with open(os.path.join(results_path, f"{task_type}_alignment.json"), "w") as f:
    #    json.dump(to_jsonable(corr_loss), f, indent = 2)
    temp = {"log": log, "epoch_acc": epoch_acc}
    with open(os.path.join(results_path, f"{task_type}_alignment.json"), "w") as f:
        json.dump(to_jsonable(temp), f, indent = 2)

    # axis removal
    print("= Axis removal =")
    acc_abl = local_axis_removal(model, test_loader, stim_test_vec, stim_test_state)

    with open(os.path.join(results_path, f"{task_type}_axis_removal.json"), "w") as f:
        json.dump(to_jsonable(acc_abl), f, indent = 2)

    # state confusion
    print("= State confusion =")
    acc_conf, state_vec_dict = local_state_confusion(task, model, test_samples, stim_test_vec, test_loader, stim_test_state)

    with open(os.path.join(results_path, f"{task_type}_state_confusion.json"), "w") as f:
        json.dump(to_jsonable(acc_conf), f, indent = 2)
    
    np.savez_compressed(os.path.join(results_path, f"{task_type}_state_vec_dict.npz"), **state_vec_dict)

Note: If you want to change the task variables, refer to building_cognitive_diagram.py
task_type: WCST
= Sample generation =
- train_samples.shape: (5000, 10) | train_ans.shape: (5000, 10)
- test_samples.shape: (1000, 10) | test_ans.shape: (1000, 10) | test_state: (1000, 10)
= Model training =
-- Early Stopped for Validation Acc >= 0.99 at epoch 7
= State confusion =


Note: If you want to change the task variables, refer to building_cognitive_diagram.py
task_type: MaxNum
= Sample generation =
- train_samples.shape: (5000, 10) | train_ans.shape: (5000, 10)
- test_samples.shape: (1000, 10) | test_ans.shape: (1000, 10) | test_state: (1000, 10)
= Model training =
-- Epoch 10 | Train Loss: 0.4165, Acc: 0.8589 | Val Loss: 0.4141, Acc: 0.8595
-- Epoch 20 | Train Loss: 0.2324, Acc: 0.9190 | Val Loss: 0.2223, Acc: 0.9253
-- Epoch 30 | Train Loss: 0.1609, Acc: 0.9533 | Val Loss: 0.1585, Acc: 0.9562
-- Epoch 40 | Train Loss: 0.1206, Acc: 0.9672 | Val Loss: 0.1082, Acc: 0.9737
-- Epoch 50 | Train Loss: 0.1016, Acc: 0.9697 | Val Loss: 0.0933, Acc: 0.9738
-- Epoch 60 | Train Loss: 0.0881, Acc: 0.9726 | Val Loss: 0.0961, Acc: 0.9735
-- Epoch 70 | Train Loss: 0.0819, Acc: 0.9743 | Val Loss: 0.0767, Acc: 0.9764
-- Epoch 80 | Train Loss: 0.0747, Acc: 0.9766 | Val Loss: 0.0733, Acc: 0.9763
-- Epoch 90 | Train Loss: 0.0713, Acc: 0.9774 | Val Loss: 0.0668, Acc: 0.9794
-

Note: If you want to change the task variables, refer to building_cognitive_diagram.py
task_type: Nback
= Sample generation =
- train_samples.shape: (5000, 10) | train_ans.shape: (5000, 10)
- test_samples.shape: (1000, 10) | test_ans.shape: (1000, 10) | test_state: (1000, 10)
= Model training =
-- Epoch 10 | Train Loss: 0.4471, Acc: 0.7667 | Val Loss: 0.4488, Acc: 0.7636
-- Epoch 20 | Train Loss: 0.2189, Acc: 0.8456 | Val Loss: 0.2172, Acc: 0.8528
-- Epoch 30 | Train Loss: 0.1534, Acc: 0.9270 | Val Loss: 0.1392, Acc: 0.9385
-- Early Stopped for Validation Acc >= 0.99 at epoch 32
= State confusion =


Note: If you want to change the task variables, refer to building_cognitive_diagram.py
task_type: FiniteSet
= Sample generation =
- train_samples.shape: (5000, 10) | train_ans.shape: (5000, 10)
- test_samples.shape: (1000, 10) | test_ans.shape: (1000, 10) | test_state: (1000, 10)
= Model training =
-- Epoch 10 | Train Loss: 0.0220, Acc: 0.9984 | Val Loss: 0.0165, Acc: 0.9984
-- Early Stopped for Validation Acc >= 0.99 at epoch 10
= State confusion =


Note: If you want to change the task variables, refer to building_cognitive_diagram.py
task_type: FiniteList
= Sample generation =
- train_samples.shape: (5000, 10) | train_ans.shape: (5000, 10)
- test_samples.shape: (1000, 10) | test_ans.shape: (1000, 10) | test_state: (1000, 10)
= Model training =
-- Epoch 10 | Train Loss: 0.1996, Acc: 0.9071 | Val Loss: 0.2015, Acc: 0.9070
-- Epoch 20 | Train Loss: 0.1781, Acc: 0.9196 | Val Loss: 0.1758, Acc: 0.9201
-- Epoch 30 | Train Loss: 0.0572, Acc: 0.9832 | Val Loss: 0.0549, Acc: 0.9832
-- Early Stopped for Validation Acc >= 0.99 at epoch 36
= State confusion =


Note: If you want to change the task variables, refer to building_cognitive_diagram.py
task_type: FiniteStack
= Sample generation =
- train_samples.shape: (5000, 10) | train_ans.shape: (5000, 10)
- test_samples.shape: (1000, 10) | test_ans.shape: (1000, 10) | test_state: (1000, 10)
= Model training =
-- Epoch 10 | Train Loss: 0.0627, Acc: 0.9795 | Val Loss: 0.0516, Acc: 0.9836
-- Early Stopped for Validation Acc >= 0.99 at epoch 14
= State confusion =
